# Day 13 — Deadlock = a Cycle in the Wait-For Graph, Visualized

A holds lock 1 wants lock 2; B holds lock 2 wants lock 1. Nobody moves. Drawn as
'who waits for whom', that's a cycle — the same cycle detection from Day 07.

In [ ]:
def find_cycle(wait_for):
    color = {u: 0 for u in set(wait_for) | {v for vs in wait_for.values() for v in vs}}
    path = []
    def dfs(u):
        color[u] = 1; path.append(u)
        for v in sorted(wait_for.get(u, []), key=str):
            if color[v] == 1: return path[path.index(v):]
            if color[v] == 0:
                r = dfs(v)
                if r: return r
        color[u] = 2; path.pop(); return None
    for u in sorted(color, key=str):
        if color[u] == 0:
            r = dfs(u)
            if r: return r
    return None

print('deadlocked:', find_cycle({'A': ['B'], 'B': ['A']}))
print('safe:      ', find_cycle({'A': ['B'], 'B': ['C'], 'C': []}))

## Reproduce, then prevent, a real two-lock deadlock

We use `acquire(timeout=...)` so the demo *reports* the deadlock instead of hanging forever.

In [ ]:
import threading, time

L1, L2 = threading.Lock(), threading.Lock()
stuck = []
def t1():
    with L1:
        time.sleep(0.1)
        if not L2.acquire(timeout=0.5): stuck.append('T1 could not get L2')
        else: L2.release()
def t2():
    with L2:
        time.sleep(0.1)
        if not L1.acquire(timeout=0.5): stuck.append('T2 could not get L1')
        else: L1.release()
a, b = threading.Thread(target=t1), threading.Thread(target=t2)
a.start(); b.start(); a.join(); b.join()
print('opposite order  ->', stuck or 'no deadlock (got lucky)')
print('fix: always acquire locks in the same global order (sorted by id/name)')

## Takeaways

- Deadlock = a cycle in the wait-for graph; detect it with DFS cycle-finding.
- Acquiring locks in one consistent global order removes the 'circular wait' condition,
  so no cycle can ever form.

**Now build** find_deadlock_cycle / has_deadlock / safe_lock_order, then `pytest -q`.